<a href="https://colab.research.google.com/github/Gallicchio-Lab/AToM-OpenMM/blob/master/example-notebooks/atom_openmm_protein_mutation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Mutation Alchemical Free Energy Calculation with AToM-OpenMM

### Acknowledgements

- OpenMM: https://openmm.org
- openmmforcefields: https://github.com/openmm/openmmforcefields
- TIAM1/Syndecan-1 system: https://doi.org/10.1021/acs.jcim.5c00207
- AToM-OpenMM theory: https://www.compmolbiophysbc.org/atom-openmm

### Procedure

- In this example, the PDB file of a receptor protein (TIAM1 PDZ domain) and the PDB files of two protein variants (wild-type and a point mutant of the Syndecan-1 peptide) are combined into a complex.
- The second variant is displaced into the solvent.
- The mutation-site backbone atoms (CA, N, C) define the alignment frame **automatically** ; no manual atom picking is needed.
- The system is solvated, minimized, thermalized, and annealed to the alchemical intermediate (λ=1/2).
- An alchemical Hamiltonian replica exchange simulation is run across 22 lambda windows.
- The perturbation energy samples are analyzed with UWHAM to obtain the relative binding free energy of the mutation (ΔΔG).

Read the [introduction to AToM-OpenMM](https://www.compmolbiophysbc.org/atom-openmm) for more background.

### To run the notebook

- Connect to a Google Colab Runtime with a GPU: click the down-arrow next to `Connect`, select `Change runtime type`, choose `T4 GPU`, then click `Connect`.
- Run the first cell to install `condacolab`. The runtime will restart — this is expected and harmless.
- After the restart, run the remaining cells in order.

### **Install Conda Colab (if on Google Colab)**

It will restart the kernel (the warning is harmless)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

### **Install dependencies**

In [ ]:
import subprocess
import sys

#fixes sys.path that gives torchvision import error
original_syspath = sys.path.copy()
new_syspath = ['/content', '/env/python', '/usr/local/lib/python312.zip', '/usr/local/lib/python3.12', '/usr/local/lib/python3.12/lib-dynload', '/usr/local/lib/python3.12/site-packages' ]
sys.path = new_syspath

#dependencies from conda-forge (no espaloma needed — protein force fields are handled by amber14)
subprocess.run("mamba install openmm openmmforcefields setproctitle -y", shell=True)

#latest atom-openmm (or 'pip install atom-openmm' for the latest released version)
subprocess.run("cd /content && git clone https://github.com/Gallicchio-Lab/AToM-OpenMM.git", shell=True)
subprocess.run("cd /content/AToM-OpenMM && pip install .", shell=True)

#installs py3Dmol for structure visualization
!pip install py3Dmol

### **Test the OpenMM installation**

In [ ]:
import openmm.testInstallation
openmm.testInstallation.main()

### **Prepare the ATM system in a box of water**

Provide the PDB files of the receptor and the two protein variants (e.g. wild-type and mutant).
Also specify the residue number where the mutation occurs and the chain name of the receptor.

In this example, the files are taken from the AToM-OpenMM repository.
To use your own system, upload the PDB files to `/content/` and update the file names and parameters below.

The job name is constructed automatically from the file names (e.g. `tiam1-sdc1wt-sdc1Q3E`).
The displacement vector (how far the second variant is moved into solvent) is also computed automatically.

In [ ]:
import os
import numpy as np
from pathlib import Path
from atom_openmm.make_atm_system_from_rcpt_lig import make_system

# ============================================================
# USER PARAMETERS  (edit these for your own system)
# ============================================================
Receptor_PDB_file_name  = 'tiam1.pdb'   #@param {type:"string"}
Variant1_PDB_file_name  = 'sdc1wt.pdb'  #@param {type:"string"}
Variant2_PDB_file_name  = 'sdc1Q3E.pdb' #@param {type:"string"}
Mutation_residue_number = 3              #@param {type:"integer"}
Receptor_chain_name     = 'B'           #@param {type:"string"}
# ============================================================

#where the receptor and variant PDB files are stored
dataDir = '/content/AToM-OpenMM/example-notebooks/data'

#full file paths
rcpt_pdb     = dataDir + '/' + Receptor_PDB_file_name
variant1_pdb = dataDir + '/' + Variant1_PDB_file_name
variant2_pdb = dataDir + '/' + Variant2_PDB_file_name

#auto-compute the displacement vector (no manual input needed)
def _get_pdb_coords(pdb_path):
    coords = []
    with open(pdb_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.array(coords, dtype=float)

def calc_displacement_vector(receptor_file, ligand2_file):
    protein_coords = _get_pdb_coords(receptor_file)
    x_range = np.array([0, protein_coords[:,0].min(), protein_coords[:,0].max()])
    y_range = np.array([1, protein_coords[:,1].min(), protein_coords[:,1].max()])
    z_range = np.array([2, protein_coords[:,2].min(), protein_coords[:,2].max()])
    axes = sorted([x_range, y_range, z_range], key=lambda v: v[2] - v[1])
    center = np.array([0., 0., 0.])
    center[int(axes[0][0])] = np.mean(axes[0][1:])
    center[int(axes[1][0])] = np.mean(axes[1][1:])
    center[int(axes[2][0])] = axes[2][2]
    u = int(axes[2][0])
    center[u] += 10.0
    lig_coords = _get_pdb_coords(ligand2_file)
    sorted_u = lig_coords[lig_coords[:, u].argsort()]
    displ = np.round(center - sorted_u[0, :], 2)
    print(f'Auto-computed displacement vector: {list(displ)} Angstroms')
    return displ

displacement = calc_displacement_vector(rcpt_pdb, variant2_pdb)

#construct job basename and working directory from file names
basename = (Path(Receptor_PDB_file_name).stem + '-' +
            Path(Variant1_PDB_file_name).stem + '-' +
            Path(Variant2_PDB_file_name).stem)

workDir = f'/content/{basename}'
os.makedirs(workDir, exist_ok=True)
os.chdir(workDir)

outxml   = f'{workDir}/{basename}_sys.xml'
pdbout   = f'{workDir}/{basename}.pdb'

print(f'Job name: {basename}')
print(f'Working directory: {workDir}')

#build the ATM system
#for protein mutation, lig1file and lig2file are PDB files (not SDF)
#amber14 force field covers protein residues automatically (no ligandforcefield override needed)
make_system(
    receptorfile = rcpt_pdb,
    lig1file     = variant1_pdb,
    lig2file     = variant2_pdb,
    displacement = list(displacement),
    xmloutfile   = outxml,
    pdboutfile   = pdbout,
    ffcachefile  = 'ff.json',
    hmass        = 1.5,
)

### **View the solvated system**

The receptor (colored by chain) is shown in the binding position.
Variant 1 (cyan) is bound in the active site; Variant 2 (orange) is displaced into solvent.

Try a different displacement vector if the position is unsatisfactory (too close to the receptor, or leading to an excessively large box).

In [ ]:
import py3Dmol

with open(pdbout, 'r') as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, 'pdb')

#receptor as cartoon, colored by spectrum
view.setStyle({'chain': Receptor_chain_name}, {'cartoon': {'color': 'spectrum'}})

#variant 1 (chain L) — bound position — cyan tube
view.setStyle({'chain': 'L'}, {'cartoon': {'color': 'cyan', 'style': 'tube'}})

#variant 2 (chain M) — displaced into solvent — orange tube
view.setStyle({'chain': 'M'}, {'cartoon': {'color': 'orange', 'style': 'tube'}})

#hide water and ions
view.setStyle({'resn': ['SOL', 'WAT', 'HOH', 'TIP3', 'Na+', 'Cl-']}, {})

view.zoomTo()
view.show()

### **Visualize the protein variants and auto-selected alignment atoms**

For protein mutation calculations, the alignment atoms are always the backbone CA, N, and C atoms at the mutation residue.
They are identified **automatically** from the residue number — no manual selection is needed.

The mutation residue is shown as sticks; the three backbone alignment atoms (CA, N, C) are shown as red spheres.

In [ ]:
#@title ##### **View Variant 1 with mutation residue highlighted**
import py3Dmol

with open(variant1_pdb, 'r') as f:
    var1_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(var1_data, 'pdb')

#show the full peptide as a tube
view.setStyle({}, {'cartoon': {'color': 'cyan', 'style': 'tube'}})

#highlight the mutation residue as sticks
view.addStyle({'resi': str(Mutation_residue_number)}, {'stick': {'radius': 0.15}})

#show backbone alignment atoms (CA, N, C) as red spheres
for atom_name in ['CA', 'N', 'C']:
    view.addStyle(
        {'resi': str(Mutation_residue_number), 'atom': atom_name},
        {'sphere': {'radius': 0.35, 'color': 'red'}}
    )

view.addLabel(
    f'Residue {Mutation_residue_number} (alignment)',
    {'resi': str(Mutation_residue_number), 'atom': 'CA'},
    {'fontColor': 'red', 'fontSize': 12, 'backgroundColor': 'white', 'backgroundOpacity': 0.7}
)

view.zoomTo({'resi': str(Mutation_residue_number)})
view.show()
print(f'Variant 1: {Variant1_PDB_file_name}')
print(f'Alignment residue: {Mutation_residue_number}  |  Atoms: CA, N, C  (auto-identified)')

In [ ]:
#@title ##### **View Variant 2 with mutation residue highlighted**
with open(variant2_pdb, 'r') as f:
    var2_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(var2_data, 'pdb')

view.setStyle({}, {'cartoon': {'color': 'orange', 'style': 'tube'}})
view.addStyle({'resi': str(Mutation_residue_number)}, {'stick': {'radius': 0.15}})

for atom_name in ['CA', 'N', 'C']:
    view.addStyle(
        {'resi': str(Mutation_residue_number), 'atom': atom_name},
        {'sphere': {'radius': 0.35, 'color': 'red'}}
    )

view.addLabel(
    f'Residue {Mutation_residue_number} (alignment)',
    {'resi': str(Mutation_residue_number), 'atom': 'CA'},
    {'fontColor': 'red', 'fontSize': 12, 'backgroundColor': 'white', 'backgroundOpacity': 0.7}
)

view.zoomTo({'resi': str(Mutation_residue_number)})
view.show()
print(f'Variant 2: {Variant2_PDB_file_name}')
print(f'Alignment residue: {Mutation_residue_number}  |  Atoms: CA, N, C  (auto-identified)')

### **Setup the protein mutation free energy calculation**

This cell:
1. Automatically identifies the CA, N, C backbone atoms at the mutation residue in each variant PDB.
2. Computes all required atom index selections (partner atoms, sidechain variable atoms, receptor frame).
3. Runs energy minimization, thermalization, and annealing to the alchemical intermediate (λ=1/2).

This takes approximately 30–60 minutes on a T4 GPU.

In [ ]:
import os
import openmm as mm
from openmm.app import *
from openmm import *
from openmm.unit import *
from atom_openmm.rbfe_structprep import rbfe_structprep
from atom_openmm.utils.AtomUtils import get_selected_principal_groups

# ── backbone atom names excluded from the alchemically variable sidechain ─────
PROTEIN_BACKBONE_ATOM_NAMES = {
    "N", "NT", "CA", "CAY", "CAT", "C", "CY", "O", "OY", "OXT",
    "H", "H2", "H3", "HY1", "HY2", "HY3", "HA",
    "DN", "DCA", "DC", "DO",
    "LPOA", "LPOB", "DOT1", "DOT2", "LPT1", "LPT2", "LPT3", "LPT4",
}

# ── topology helper functions (adapted from run-atm.py) ───────────────────────
def get_indexes_from_query(topology, query):
    indexes = [ atom.index for atom in topology.atoms() if eval(query, {"atom": atom}) ]
    indexes.sort()
    return indexes

def get_indexes_from_residue(residue, query='True'):
    indexes = [
        atom.index for atom in residue.atoms()
        if eval(query, {"atom": atom, "PROTEIN_BACKBONE_ATOM_NAMES": PROTEIN_BACKBONE_ATOM_NAMES})
    ]
    indexes.sort()
    return indexes

def get_indexes_from_residues(residues, query='True'):
    indexes = []
    for residue in residues:
        indexes.extend(
            atom.index for atom in residue.atoms()
            if eval(query, {"atom": atom, "PROTEIN_BACKBONE_ATOM_NAMES": PROTEIN_BACKBONE_ATOM_NAMES})
        )
    indexes.sort()
    return indexes

def get_residues_by_name(topology, residue_name):
    return [r for chain in topology.chains() for r in chain.residues() if r.name == residue_name]

def get_residues_by_chain_id(topology, chain_id):
    return [r for chain in topology.chains() if chain.id == chain_id for r in chain.residues()]

def get_residue_by_id(residues, residue_id):
    matches = [r for r in residues if r.id == str(residue_id)]
    if not matches:
        return None
    if len(matches) > 1:
        raise ValueError(f"Multiple residues with id {residue_id}")
    return matches[0]

def cm_from_indexes(topology, positions, indexes):
    cm = Vec3(0, 0, 0) * nanometer
    n = 0
    for atom in topology.atoms():
        if atom.index in indexes:
            cm += positions[atom.index]
            n += 1
    return cm / float(n)

# ── auto-detect backbone alignment atoms (CA/N/C) at the mutation residue ─────
ATOM_ORDER = ('CA', 'N', 'C')

def collect_backbone_atom_ids(pdb_path, residue_id):
    """Return 1-based PDB serial indices [CA_idx, N_idx, C_idx] at residue_id."""
    atoms_by_name = {}
    atom_index = 0
    preferred = {'': 0, 'A': 1}
    with open(pdb_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if not (line.startswith('ATOM') or line.startswith('HETATM')):
                continue
            atom_index += 1
            atom_name = line[12:16].strip()
            altloc     = line[16].strip()
            resid      = line[22:26].strip()
            if resid != str(residue_id):
                continue
            if atom_name not in ATOM_ORDER:
                continue
            existing = atoms_by_name.get(atom_name)
            if existing is None or preferred.get(altloc, 2) < preferred.get(existing['altloc'], 2):
                atoms_by_name[atom_name] = {'index': atom_index, 'altloc': altloc}
    missing = [n for n in ATOM_ORDER if n not in atoms_by_name]
    if missing:
        raise ValueError(f'{pdb_path}: residue {residue_id} is missing backbone atoms {missing}')
    return [atoms_by_name[n]['index'] for n in ATOM_ORDER]

#find 1-based alignment atom IDs in each variant
lig1_align_ids_1based = collect_backbone_atom_ids(variant1_pdb, Mutation_residue_number)
lig2_align_ids_1based = collect_backbone_atom_ids(variant2_pdb, Mutation_residue_number)
print(f'Variant 1 alignment atom IDs (1-based PDB serials, CA/N/C): {lig1_align_ids_1based}')
print(f'Variant 2 alignment atom IDs (1-based PDB serials, CA/N/C): {lig2_align_ids_1based}')

#convert to 0-based indices (as expected by the AToM options dict)
lig1_ref_atoms_0based = [i - 1 for i in lig1_align_ids_1based]
lig2_ref_atoms_0based = [i - 1 for i in lig2_align_ids_1based]

# ── alchemical schedule and simulation parameters ─────────────────────────────
options = {
        "JOB_TRANSPORT": 'LOCAL_OPENMM',
        "RE_SETUP": 'YES',
        "TEMPERATURES": [ 310.0 ],   #body temperature (K)
        "LAMBDAS":      [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00],
        "DIRECTION":    [   1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1],
        "INTERMEDIATE": [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    1,    1,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        "LAMBDA1":      [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.50, 0.40, 0.30, 0.20, 0.10, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        "LAMBDA2":      [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.40, 0.30, 0.20, 0.10, 0.00],
        "ALPHA":        [0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10],
        "U0":           [110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110.],
        "W0COEFF":      [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        "WALL_TIME": 240,      #max simulation time in minutes (reduced for Colab)
        "MAX_SAMPLES": 10,     #samples per replica to collect (use >=100 for production)
        "CYCLE_TIME": 10,      #interval between replica-exchange attempts in seconds
        "CHECKPOINT_TIME": 1200,  #interval between checkpoints in seconds
        "SUBJOBS_BUFFER_SIZE": 1.0,
        "PRODUCTION_STEPS": 10000, #MD steps per replica per cycle (~40 ps)
        "PRNT_FREQUENCY":   10000,
        "TRJ_FREQUENCY":    20000,
        "CM_KF": 25.00,        #force constant of flat-bottom binding site restraint (kcal/mol/Ang^2)
        "CM_TOL": 5,           #radius of binding site restraint (Ang)
        "POSRE_FORCE_CONSTANT": 25.0,  #position restraints ON for protein mutation (kcal/mol/Ang^2)
        "POSRE_TOLERANCE": 3.5,
        "ALIGN_KF_SEP": 0.0,   #zero = variable displacement mode (appropriate for protein mutation)
        "ALIGN_K_THETA": 25.0, #orientation alignment force constant (kcal/mol)
        "ALIGN_K_PSI":   25.0, #roll alignment force constant (kcal/mol)
        "UMAX": 200.00,        #max energy parameter of the soft-core potential (kcal/mol)
        "ACORE": 0.062500,     #a-parameter of the soft-core potential
        "UBCORE": 100.0,       #ub-parameter of the soft-core potential (kcal/mol)
        "FRICTION_COEFF": 0.500000,  #Langevin friction coefficient (ps)
        "TIME_STEP": 0.004,    #MD timestep (ps) — 4 fs with hydrogen mass repartitioning
        "HMASS": 1.5,          #hydrogen mass repartitioning (amu)
        "VERBOSE": False,
}

#identity and workflow fields
options['BASENAME']                  = basename
options['WORKFLOW_MODE']             = 'protein_mutation'
options['RCPT_CHAIN_NAME']           = Receptor_chain_name
options['PAIR_ALIGNMENT_RESIDUE']    = str(Mutation_residue_number)
options['PAIR_ALIGNMENT_ATOM_ORDER'] = list(ATOM_ORDER)
options['ALIGN_LIGAND1_REF_ATOMS']   = lig1_ref_atoms_0based
options['ALIGN_LIGAND2_REF_ATOMS']   = lig2_ref_atoms_0based
options['DISPLACEMENT']              = list(displacement)

#load the prepared system PDB
pdb      = PDBFile(pdbout)
topology = pdb.topology
positions = pdb.positions

# ── partner 1 atoms (residue name L1, fallback chain L) ──────────────────────
res1 = get_residues_by_name(topology, 'L1')
if not res1:
    res1 = get_residues_by_chain_id(topology, 'L')
if not res1:
    raise ValueError("Cannot find variant 1 in prepared PDB (expected residue L1 or chain L)")
ligand1_atoms = get_indexes_from_residues(res1)
options['LIGAND1_ATOMS'] = ligand1_atoms

# ── partner 2 atoms (residue name L2, fallback chain M) ──────────────────────
res2 = get_residues_by_name(topology, 'L2')
if not res2:
    res2 = get_residues_by_chain_id(topology, 'M')
if not res2:
    raise ValueError("Cannot find variant 2 in prepared PDB (expected residue L2 or chain M)")
ligand2_atoms = get_indexes_from_residues(res2)
options['LIGAND2_ATOMS'] = ligand2_atoms

# ── variable atoms = sidechain only at the mutation residue (not backbone) ────
res1_var = get_residue_by_id(res1, Mutation_residue_number)
res2_var = get_residue_by_id(res2, Mutation_residue_number)
if res1_var is None:
    raise ValueError(f"Mutation residue {Mutation_residue_number} not found in variant 1")
if res2_var is None:
    raise ValueError(f"Mutation residue {Mutation_residue_number} not found in variant 2")

options['LIGAND1_VAR_ATOMS'] = get_indexes_from_residue(
    res1_var, query='atom.name not in PROTEIN_BACKBONE_ATOM_NAMES'
)
options['LIGAND2_VAR_ATOMS'] = get_indexes_from_residue(
    res2_var, query='atom.name not in PROTEIN_BACKBONE_ATOM_NAMES'
)

# ── attachment atom = CA of the mutation residue ──────────────────────────────
lig1_anchor = ligand1_atoms[lig1_ref_atoms_0based[0]]
lig2_anchor = ligand2_atoms[lig2_ref_atoms_0based[0]]
options['LIGAND1_ATTACH_ATOM'] = lig1_anchor
options['LIGAND2_ATTACH_ATOM'] = lig2_anchor

#CM atoms = attachment atom (CA of mutation residue)
options['LIGAND1_CM_ATOMS'] = [lig1_anchor]
options['LIGAND2_CM_ATOMS'] = [lig2_anchor]

#positions of the CMs
lig1cm = cm_from_indexes(topology, positions, options['LIGAND1_CM_ATOMS'])
lig2cm = cm_from_indexes(topology, positions, options['LIGAND2_CM_ATOMS'])

#actual displacement from the prepared PDB geometry (overrides the initial estimate)
displ = (lig2cm - lig1cm).value_in_unit(angstrom)
options['DISPLACEMENT'] = [displ.x, displ.y, displ.z]
print(f'Actual displacement from PDB geometry: {[round(v, 2) for v in options["DISPLACEMENT"]]} Angstroms')

# ── receptor reference frame (CA atoms of receptor chain) ─────────────────────
rcpt_query = f'atom.residue.chain.id == "{Receptor_chain_name}" and atom.name == "CA"'
rcpt_frame_indexes = get_indexes_from_query(topology, rcpt_query)
rcpt_frame = get_selected_principal_groups(topology, positions, rcpt_frame_indexes)

options['RCPT_CM_ATOMS']      = rcpt_frame['origin']['indices']
options['RCPT_FRAME_ATOMS_O'] = options['RCPT_CM_ATOMS']
options['RCPT_FRAME_ATOMS_Z'] = rcpt_frame['z_axis']['indices']
options['RCPT_FRAME_ATOMS_Y'] = rcpt_frame['y_axis']['indices']

#offset of variant 1 CM from the receptor CM
rcpt_cm_pos = Vec3(float(rcpt_frame['origin']['com'][0]),
                   float(rcpt_frame['origin']['com'][1]),
                   float(rcpt_frame['origin']['com'][2])) * nanometer
offset = (lig1cm - rcpt_cm_pos).value_in_unit(angstrom)
options['LIGOFFSET'] = [offset.x, offset.y, offset.z]

#position-restrained atoms (receptor CA atoms)
options['POS_RESTRAINED_ATOMS'] = options['RCPT_CM_ATOMS']

# ── exclusion potential: keeps displaced variant away from receptor ────────────
options['EXCLUSION_POT_MOL1_INDEXES'] = get_indexes_from_query(
    topology,
    f'atom.residue.chain.id == "{Receptor_chain_name}" and atom.element.atomic_number != 1'
)
options['EXCLUSION_POT_MOL2_INDEXES'] = get_indexes_from_residues(
    res2, query='atom.element.atomic_number != 1'
)

#working directory
options['WORKDIR'] = workDir

print()
print("=== ATM PROTEIN MUTATION SETTINGS ===")
for k in ['BASENAME', 'WORKFLOW_MODE', 'RCPT_CHAIN_NAME', 'PAIR_ALIGNMENT_RESIDUE',
          'ALIGN_LIGAND1_REF_ATOMS', 'ALIGN_LIGAND2_REF_ATOMS',
          'DISPLACEMENT', 'LIGAND1_ATTACH_ATOM', 'LIGAND2_ATTACH_ATOM',
          'POSRE_FORCE_CONSTANT', 'ALIGN_KF_SEP', 'TEMPERATURES']:
    print(f'  {k}: {options[k]}')

#run energy minimization, thermalization, and annealing to lambda=0.5
rbfe_structprep(config_file=None, options=options)

### **Run the protein mutation free energy simulation**

Hamiltonian replica exchange across 22 lambda windows.
With MAX_SAMPLES=10 and a T4 GPU this takes approximately 1–2 hours.
The simulation checkpoints every 20 minutes — if interrupted, re-running this cell will resume from the last checkpoint.

In [ ]:
from atom_openmm.rbfe_production import rbfe_production
rbfe_production(config_file=None, options=options)

### **Analysis: relative binding free energy estimate**

UWHAM analysis on the collected perturbation energy samples.
The first third of samples is discarded for equilibration.

**Note:** 10 samples per replica is a tutorial-level run. Reliable estimates require ≥100 samples.
A positive ΔΔG means the mutation weakens binding (Variant 2 binds more weakly than Variant 1).

In [ ]:
from atom_openmm.uwham import calculate_uwham_from_rundir, create_quality_assessment_plot

ddG, ddG_std, uwham_data = calculate_uwham_from_rundir(
    options['WORKDIR'], options['BASENAME']
)
print(f'DDGb = {ddG:.3f} +/- {ddG_std:.3f} kcal/mol')
print(f'(positive DDGb = variant 2 binds more weakly than variant 1)')

#quality assessment plot
df1 = uwham_data['df_leg1']
N = len(uwham_data['uwham_out_leg1']['W'][:,0])
df1['W'] = uwham_data['uwham_out_leg1']['W'][:,0] / float(N)
df2 = uwham_data['df_leg2']
N = len(uwham_data['uwham_out_leg2']['W'][:,0])
df2['W'] = uwham_data['uwham_out_leg2']['W'][:,0] / float(N)
fig = create_quality_assessment_plot(df1, df2)

### **Download the results as a zip file**

In [ ]:
from google.colab import files
print(f'Packaging {basename}.zip ...')
!cd /content && zip -r {basename}.zip {basename}
files.download(f'/content/{basename}.zip')